# Metropolis–Hastings MCMC

Chapter 18: Langevin Dynamics and Sampling · *Mathematics for AI and Machine Learning* by Fuheng Wu

Sample a bimodal distribution with random-walk Metropolis–Hastings.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-18/metropolis-hastings/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: target p(x) ∝ exp(-(x+2)^2/2) + exp(-(x-2)^2/2), Gaussian random walk
steps, sigma = 6000, 0.5


def log_target(x):
    # log(e^a + e^b) computed stably, so far-away proposals don't underflow to log(0)
    a, b = -0.5 * (x + 2) ** 2, -0.5 * (x - 2) ** 2
    hi = max(a, b)
    return hi + math.log(math.exp(a - hi) + math.exp(b - hi))


def metropolis(steps, sigma, x=0.0):
    props = torch.randn(steps).tolist()
    us = torch.rand(steps).tolist()
    out, accepted = [], 0
    for z, u in zip(props, us):
        y = x + sigma * z
        # Symmetric proposal: accept with probability min(1, p(y) / p(x))
        if math.log(u) <= log_target(y) - log_target(x):
            x, accepted = y, accepted + 1
        out.append(x)
    return torch.tensor(out), accepted / steps


samples, rate = metropolis(steps, sigma)
print(f"sigma = {sigma}: acceptance {rate:.3f}, mean {samples.mean().item():+.3f} (0), "
      f"variance {samples.var().item():.3f} (target 5), fraction x > 0: {(samples > 0).float().mean().item():.3f}")

# Count jumps between the two modes
side = (samples > 0).int()
print("mode switches in the chain:", int((side[1:] != side[:-1]).sum()))

# ── Proposal width: too small creeps, too big gets rejected ─────────────────
print("\n sigma   acceptance   mode switches   variance (target 5)")
for s in (0.1, 0.5, 2.0, 4.0, 15.0):
    sm, r = metropolis(20000, s)
    sd = (sm > 0).int()
    print(f"{s:5.1f}     {r:.3f}        {int((sd[1:] != sd[:-1]).sum()):5d}          {sm.var().item():.3f}")

sigma = 0.5: acceptance 0.873, mean -0.101 (0), variance 4.639 (target 5), fraction x > 0: 0.478
mode switches in the chain: 190

 sigma   acceptance   mode switches   variance (target 5)
  0.1     0.969          129          4.623
  0.5     0.865          478          4.979
  2.0     0.632         2668          5.009
  4.0     0.471         3759          5.172
 15.0     0.168         1682          5.006
